# ClimateFund QA modular notebook flow

This notebook uses the same function-based pipeline that can also be run from CMD with `python -m climatefund_qa.cli ...`.

The flow is intentionally close to the original experiment notebook:

1. install/import
2. configuration and credentials
3. prepare document/passages tables
4. build/load indexes
5. retrieve with or without reranking
6. reader answer generation
7. full experiment run
8. main results table and qualitative examples


In [ ]:
# ============================================================
# 0. Install dependencies, if needed
# ============================================================
# Run this once in a fresh environment.
# !pip install -r ../requirements.txt


In [ ]:
# ============================================================
# 1. Make sure Python can import climatefund_qa
# ============================================================

from pathlib import Path
import sys

# If this notebook is inside UOG/code/notebooks, project code folder is UOG/code.
NOTEBOOK_DIR = Path.cwd()
CODE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("CODE_DIR:", CODE_DIR)
print("Contains package:", (CODE_DIR / "climatefund_qa").exists())

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


In [ ]:
# ============================================================
# 2. Imports and configuration
# ============================================================

from climatefund_qa import (
    create_config,
    step_status,
    step_prepare_tables,
    step_build_indexes,
    step_load_indexes,
    step_load_or_build_indexes,
    step_retrieve,
    step_read_answer,
    step_run_experiment,
)
from climatefund_qa.credentials import load_credentials, credential_status
from climatefund_qa.results_table import create_and_save_main_results
from climatefund_qa.qualitative import create_qualitative_examples, load_latest_run_outputs, print_example

# Project layout expected:
# UOG/
# ├── code/
# │   └── notebooks/
# └── dataset/
#     ├── climatefund_qa.csv
#     └── documents/

PROJECT_ROOT = CODE_DIR.parent

cfg = create_config(
    project_root=PROJECT_ROOT,
    dataset_filename="climatefund_qa.csv",
)

# Keep this close to the original notebook settings.
cfg.retrievers = ["bm25", "e5"]
cfg.rerankers = ["none", "t5"]
cfg.readers = ["extractive_fallback"]  # safe smoke-test reader; replace with real reader IDs later

cfg.retrieve_k = 50
cfg.rerank_input_k = 20
cfg.final_context_k = 5
cfg.max_questions = None
cfg.compute_bertscore = True

# Rebuild switches. First run can use True; later use False for speed.
cfg.rebuild_document_tables = True
cfg.rebuild_passage_table = True
cfg.rebuild_bm25_passage_index = True
cfg.rebuild_e5_passage_index = True

load_credentials()
print("Credential status:", credential_status())
step_status(cfg)


In [ ]:
# ============================================================
# 3. Prepare document, passage, and question tables
# ============================================================
# This corresponds to the original notebook cells that load PDFs/TXT,
# create documents.csv, create passages.csv, and load the QA dataset.

tables = step_prepare_tables(cfg)

docs_df = tables["docs_df"]
passages_df = tables["passages_df"]
qa_df = tables["qa_df"]

display(docs_df.head())
display(passages_df.head())
display(qa_df.head())


In [ ]:
# ============================================================
# 4. Build or load indexes
# ============================================================
# This corresponds to the original BM25/E5 global + per-project index cells.
# Use retrievers=["bm25"] for faster testing.

indexes = step_load_or_build_indexes(
    cfg,
    passages_df,
    retrievers=["bm25"],       # change to ["bm25", "e5"] when ready
    rebuild=True,              # set False after indexes exist
)

print(indexes.keys())
print("Global indexes:", indexes["global"].keys())
print("Project BM25 indexes:", len(indexes["by_project"].get("bm25", {})))


In [ ]:
# ============================================================
# 5. Manual retrieval test: no reranker
# ============================================================
# This is useful before running the full grid.

question = "What is the project objective?"

retrieved = step_retrieve(
    cfg,
    question=question,
    indexes=indexes,
    passages_df=passages_df,
    retriever="bm25",
    reranker="none",
    scope="cross_projects",
)

display(retrieved[[c for c in ["rank", "docno", "project", "score", "text"] if c in retrieved.columns]].head())


In [ ]:
# ============================================================
# 6. Manual reader test
# ============================================================
# extractive_fallback works without API keys.
# Later you can use real readers such as openai_gpt_4o_mini or ida_qwen_2_5_72b.

answer = step_read_answer(
    cfg,
    question=question,
    contexts=retrieved,
    reader="extractive_fallback",
)

print(answer)


In [ ]:
# ============================================================
# 7. Run a small experiment grid
# ============================================================
# Start small. Then increase max_questions and add e5/t5/real readers.

results = step_run_experiment(
    cfg,
    retrievers=["bm25"],
    rerankers=["none"],
    readers=["extractive_fallback"],
    max_questions=3,
    rebuild_indexes=False,
    compute_bertscore_flag=False,
)

print("Output directory:", results["output_dir"])
display(results["metrics_df"].head())
display(results["predictions_df"].head())


In [ ]:
# ============================================================
# 8. Full experiment example
# ============================================================
# Uncomment when ready.

# results = step_run_experiment(
#     cfg,
#     retrievers=["bm25", "e5"],
#     rerankers=["none", "t5"],
#     readers=["hf_mistral_7b_instruct_v03"],
#     max_questions=None,
#     rebuild_indexes=False,
#     compute_bertscore_flag=True,
# )


In [ ]:
# ============================================================
# 9. Main results table
# ============================================================

latest_run_dir, final_table_df, latex_main_results = create_and_save_main_results(
    cfg,
    table_retrievers=None,
    table_rerankers=None,
    table_readers=None,
)

print("Using run directory:", latest_run_dir)
display(final_table_df)
print(latex_main_results)


In [ ]:
# ============================================================
# 10. Qualitative examples
# ============================================================

latest_run_dir, examples = create_qualitative_examples(cfg, top_k=5, n=2)
print("Qualitative examples saved to:", latest_run_dir / "qualitative_examples")
print(examples.keys())
